### 4.1.1 Visualização das séries temporais de cada estação

**Figura XXX - Disposição espacial das estações de monitoramento da qualidade do ar. Séries temporais para cada uma das estações dentro dos popups**

In [2]:
import os
import folium
import pandas as pd
import base64
import numpy as np
import warnings
import scripts.flagTables as flagtab
warnings.filterwarnings('ignore')

# Caminho para a pasta de dados (exemplo)
rootPath = os.path.dirname(os.getcwd())

# Lendo o csv
aqmData = pd.read_csv(rootPath+'/data/Monitoramento_QAr_BR.csv')
aqmData['STATUS'] = aqmData['STATUS'].replace(np.nan, 'Desconhecido', regex=True)
aqmData['CATEGORIA'] = aqmData['CATEGORIA'].replace(np.nan, 'Desconhecida', regex=True)
aqmData.loc[aqmData['ID_OEMA'].isna(),'ID_OEMA'] = '-'
aqmData.loc[aqmData['CIDADE'].isna(),'CIDADE'] = '-'
aqmData.loc[aqmData['ID_MMA_COMPLETO'].isna(),'ID_MMA_COMPLETO'] = '-'

columnsSelector = ["","UF","ID_OEMA", "Status","Poluente",'Base de dados']
aqmData = flagtab.flagTable(columnsSelector)
searchPaneColumns=[1,4,5]
flagtab.tabela_iterativa(aqmData,searchPaneColumns)


Loading ITables v2.5.1 from the init_notebook_mode cell... (need help?)


In [6]:
# Caminho para a pasta de dados (exemplo)
rootPath = os.path.dirname(os.getcwd())

# Lendo o csv
aqmData = pd.read_csv(rootPath+'/data/Monitoramento_QAr_BR.csv')
aqmData['STATUS'] = aqmData['STATUS'].replace(np.nan, 'Desconhecido', regex=True)
aqmData['CATEGORIA'] = aqmData['CATEGORIA'].replace(np.nan, 'Desconhecida', regex=True)
aqmData.loc[aqmData['ID_OEMA'].isna(),'ID_OEMA'] = '-'
aqmData.loc[aqmData['CIDADE'].isna(),'CIDADE'] = '-'
aqmData.loc[aqmData['ID_MMA_COMPLETO'].isna(),'ID_MMA_COMPLETO'] = '-'
# Folium section
aqmData['status_category'] = aqmData['STATUS'].apply(lambda x: x.split('\n')[0][0].upper())+aqmData['CATEGORIA'].apply(lambda x: x.split('\n')[0][0].upper())
color_map = {
    'AR': '#038cfc',
    'IR': '#c8f2fa',
    'AI': '#f08432',
    'II': '#ffea8f'
}
aqmData['COD_POLUENTE'] = aqmData['COD_POLUENTE'].astype(str)
aqmData = aqmData.fillna('-')
remaining_columns = aqmData.columns[(aqmData.columns != 'POLUENTE') & (aqmData.columns != 'ID_MMA_COMPLETO') & (aqmData.columns != 'COD_POLUENTE')].tolist()
#print(remaining_columns)

# Agrupamento por estado quando tivermos mais de uma fonte de informação
aqmDataGrouped = aqmData.groupby(remaining_columns).agg({
    'POLUENTE': lambda x: ','.join(x),
    'ID_MMA_COMPLETO': lambda x: ','.join(x),
    'COD_POLUENTE': lambda x: ','.join(x),
}).reset_index()
#print(aqmDataGrouped)



# Criando o mapa centralizado na média das coordenadas
map_clusters = folium.Map(
    location=[aqmDataGrouped['LATITUDE'].mean(), aqmDataGrouped['LONGITUDE'].mean()],
    tiles="OpenStreetMap",
    zoom_start=5
)

brazil_bounds = [[-33.75, -73.98], [5.27, -34.79]]
map_clusters.fit_bounds(brazil_bounds)
map_clusters.options['maxBounds'] = brazil_bounds


# Removendo linhas com latitude ou longitude faltantes
aqmDataGrouped = aqmDataGrouped.dropna(subset=['LATITUDE', 'LONGITUDE'])


# Adiciona os marcadores com o popup que exibe o botão para mostrar o iframe
for index, row in aqmDataGrouped.iterrows():
    popup_html = """
        ID_MMA: """+row.ID_MMA +\
        """<br>ID_OEMA: """+row.ID_OEMA +\
        """<br>Cidade: """+row.CIDADE
    
    popup_pol = ''
    for ii, pol in enumerate(row.POLUENTE.split(',')):
        if (row.BASE_DADOS) & (row.ID_MMA_COMPLETO !=''):
            popup_pol = popup_pol + """<br><a href="""+'../_static/plotly_figures/timeSeriesFigures/'+row.ID_MMA_COMPLETO.split(',')[ii]+'.html'+""" target="_blank">Série temporal de """+ pol + """ </a> """
        
    popup = folium.Popup(popup_html+popup_pol, max_width=1000, max_height=600,)
    
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=4,
        fill=True,
        fill_opacity=0.7,
        popup=popup,
        color=color_map.get(row['status_category'], 'gray'),
        fill_color=color_map.get(row['status_category'], 'gray')
    ).add_to(map_clusters)
    # Add legend
    legend_html = """
    <div style="
        position: fixed;
        bottom: 50px;
        left: 50px;
        width: 160px;
        background-color: white;
        border:2px solid grey;
        z-index:9999;
        font-size:14px;
        padding: 10px;
        ">
    <b>Legenda</b><br>
    <i style="background:#038cfc; width:10px; height:10px; float:left; margin-right:5px; opacity:0.9;"></i> Referência/Ativa<br>
    <i style="background:#c8f2fa; width:10px; height:10px; float:left; margin-right:5px; opacity:0.9;"></i> Referência/Inativa<br>
    <i style="background:#f08432; width:10px; height:10px; float:left; margin-right:5px; opacity:0.9;"></i> Indicativa/Ativa<br>
    <i style="background:#ffea8f; width:10px; height:10px; float:left; margin-right:5px; opacity:0.9;"></i> Indicativa/Inativa<br>
    </div>
    """
    map_clusters.get_root().html.add_child(folium.Element(legend_html))

map_clusters

